# module-composition — worked example 1: Two named child Linears auto-register

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `module-composition`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Assigning a child `nn.Module` as an attribute (`self.fc1 = nn.Linear(...)`) auto-registers it as a submodule. The parent's `.parameters()` then transitively includes the children's parameters, and `.named_modules()` shows the children under the attribute names you chose.

## Worked solution

We build the simplest composition: a parent holding two named Linear children.

1. **super().__init__() first.** This sets up the internal `_modules` dict; assigning children before it raises.
2. **Name the children.** `self.fc1` and `self.fc2` become registered submodules keyed by those names. The names propagate into parameter names like `fc1.weight`.
3. **forward composition.** `fc2(relu(fc1(x)))` chains the two layers with a ReLU between.
4. **Transitivity.** The parent's `.parameters()` yields all four tensors (two weights, two biases) because registration recurses.

The demo builds the MLP, prints the child names from `named_children()`, the total parameter count (4), and the output shape for a batch.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(0)

class TwoLayerMLP(nn.Module):
    def __init__(self, in_f, hid, out_f):
        super().__init__()
        self.fc1 = nn.Linear(in_f, hid)
        self.fc2 = nn.Linear(hid, out_f)
    def forward(self, x):
        return self.fc2(t.relu(self.fc1(x)))

model = TwoLayerMLP(4, 8, 3)
print('child names:', [n for n, _ in model.named_children()])
print('param count:', sum(1 for _ in model.parameters()))
print('output shape:', tuple(model(t.randn(5, 4)).shape))